This notebook is tuned for a single **NVIDIA A100 40GB** (SXM4). For the original T4 / free-Colab version, see the upstream Unsloth notebook.
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

# Goal: Fine-tune Gemma 4 on the `direct_debit_explainer` API with RLVR

A production `direct_debit_explainer` API is served by gemini-2.5-pro today. Given a customer's `account_context` and the `latest_dd_change`, it returns a list of `TriggerExplanation`s — one per relevant reason the direct debit changed, drawn from a closed 7-value `Trigger` enum.

Online evaluation of that API flagged a material failure rate (up to ~35% on a representative 7-day window) across three categories: incorrectly identified previous-DD amounts, hallucinated tariff names or rate percentages, and inappropriate use of "underpayment" language. All three are **verifiable** — you can check them against the input.

In this notebook we GRPO-fine-tune Gemma 4 against the same I/O contract using:
- a synthetic `DDExplainerPromptInput` dataset with ground-truth trigger labels (from `dd_explainer.py`),
- reward functions that encode both the general domain rules (trigger detection via `detect_triggers`) and the three specific failure categories from the production error analysis.

# Installation
We'll be using [Unsloth](https://github.com/unslothai/unsloth) to do RL on Gemma 4. Unsloth saves 70% VRAM usage and makes reinforcement learning 2 to 6x faster.

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    # Gemma 4 requires transformers >= 5.5.0 — do NOT pin to 4.x here
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
# Gemma 4 requires transformers >= 5.5.0
!uv pip install --upgrade --no-deps "transformers>=5.5.0" tokenizers "trl>=0.28.0" unsloth unsloth_zoo

In [ ]:
%%capture
!pip install --no-deps --upgrade timm # For Gemma 4 vision/audio

In [ ]:
# Redirect heavy caches + checkpoints to the /workspace NVMe volume (195GB)
# instead of the overlay root. MUST run before any HF / torch import.
import os
WORKSPACE = "/workspace/gemma4_rl"
os.makedirs(f"{WORKSPACE}/hf_cache", exist_ok=True)
os.makedirs(f"{WORKSPACE}/torch_cache", exist_ok=True)
os.makedirs(f"{WORKSPACE}/outputs", exist_ok=True)

os.environ["HF_HOME"]           = f"{WORKSPACE}/hf_cache"
os.environ["HF_DATASETS_CACHE"] = f"{WORKSPACE}/hf_cache/datasets"
os.environ["TORCH_HOME"]        = f"{WORKSPACE}/torch_cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"  # faster model downloads
os.environ["TORCHINDUCTOR_CACHE_DIR"] = f"{WORKSPACE}/torch_cache/inductor"  # persist compiled Triton kernels on NVMe

### Unsloth

In [ ]:
from unsloth import FastVisionModel
import torch
# DD explainer prompt is ~3.1k tokens (domain knowledge + input JSON); bump from 4096 so completions have room.
max_seq_length = 6144
lora_rank = 64         # A100 40GB easily fits 64; 128 also viable but slower.

gemma4_models = [
    # Gemma-4 instruct models:
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
    # Gemma-4 base models:
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    max_seq_length = max_seq_length,
    load_in_4bit = False,   # A100 40GB has room for 16-bit LoRA — better gradients than QLoRA.
    fast_inference = False, # vLLM path is unstable for Gemma 4 multimodal; keep False until tested.
)

To do efficient RL, we will use [LoRA](https://arxiv.org/abs/2106.09685), which allows us to only add 1 to 5% of extra weights to the model for finetuning purposes. This allows us to save memory usage by over 60%, and yet it retains good accuracy.

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = False, # Was "unsloth"; disabled because it forces past_key_values=None in Gemma4 generation — KV cache off made generation ~5-10× slower. A100 40GB has ample memory.
    random_state = 3407,
)

# Synthetic DD explainer dataset

The input schema (`DDExplainerPromptInput`), output schema (`DirectDebitExplainerResponse`), domain knowledge, and scenario-first synthetic data generator all live in `dd_explainer.py`. Each generated example picks a target set of `Trigger`s and constructs an `AccountContext` whose data satisfies the domain rules for exactly that set — giving us a ground-truth label for every training example.

In [ ]:
import sys
sys.path.insert(0, "/workspace/gemma4_rl")

from dd_explainer_data_generator import (
    DDExplainerPromptInput,
    DirectDebitExplainerResponse,
    Trigger,
    TriggerExplanation,
    detect_triggers,
    generate_dd_example,
    build_dataset,
    build_chat_messages,
    SYSTEM_PROMPT,
    DOMAIN_KNOWLEDGE,
)

print("Triggers in the closed enum:")
for t in Trigger:
    print(f"  - {t.value}")

Generate one example and inspect it — a single `DDExplainerPromptInput` plus the target triggers the generator was asked to produce. `detect_triggers` is the rule-based oracle; when we call it on the freshly generated input, it should recover the exact target set.

In [ ]:
import random, json

rng = random.Random(7)
target = {Trigger.Change_in_usage, Trigger.Missed_bounced_DD_payments}
example = generate_dd_example(target, rng)

print(f"Target triggers: {sorted(t.value for t in target)}")
print(f"Oracle recovered: {sorted(t.value for t in detect_triggers(example))}")
print()
print("latest_dd_change:")
print(json.dumps(example.latest_dd_change.model_dump(mode='json'), indent=2, default=str))

# Prompt & baseline rollout

The prompt template is the same chat schema the production `direct_debit_explainer` API uses (system message with domain knowledge + user message with the rendered `latest_dd_change` and `account_context`). `dd_explainer.build_chat_messages` renders one for the example above.

In [ ]:
messages = build_chat_messages(example)

total_chars = sum(len(m["content"]) for m in messages)
print(f"Chat messages: {len(messages)} ({total_chars} chars total)")
print("---- system (truncated) ----")
print(messages[0]["content"][:400], "...\n")
print("---- user (truncated) ----")
print(messages[1]["content"][:800], "...\n")

First, let's prompt the model without RL and see how it goes:

In [ ]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
)

from transformers import TextStreamer
print("=" * 60)
print("BASE MODEL OUTPUT (before RL training):")
print("=" * 60)

inputs = tokenizer(
    text = text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 1.0, top_p = 0.95, top_k = 64,
)

# Reward functions

Seven verifiable rewards, grouped by purpose:

1. **Shape** — `extract_json` + `reward_schema_valid` + `reward_triggers_in_enum`: output must parse as `DirectDebitExplainerResponse`, and each `trigger` must be a valid enum value.
2. **Correctness** — `reward_triggers_match_ground_truth`: F1 between predicted and oracle trigger sets. This is the main RL signal.
3. **Production failure modes** — `reward_previous_dd_amount_correct`, `reward_no_hallucinated_facts`, `reward_underpayment_language_constrained`: one per category identified in the LangSmith error-analysis reports.
4. **Well-formed explanations** — `reward_explanations_well_formed`: shape/length check.

In [ ]:
import re
from typing import Optional

_JSON_FENCE_RE = re.compile(r"```(?:json)?\s*(\{.*?\})\s*```", re.DOTALL)

def extract_json(text: str) -> Optional[str]:
    """Return the first JSON object string found in `text`, stripping markdown fences."""
    m = _JSON_FENCE_RE.search(text)
    if m:
        return m.group(1)
    # Bare JSON fallback: find outermost {...}.
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        c = text[i]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return None


def _parse_response(text: str) -> Optional[DirectDebitExplainerResponse]:
    raw = extract_json(text)
    if raw is None:
        return None
    try:
        return DirectDebitExplainerResponse.model_validate_json(raw)
    except Exception:
        return None


def reward_schema_valid(completions, **kwargs):
    scores = []
    for c in completions:
        parsed = _parse_response(c[0]["content"])
        scores.append(1.0 if parsed is not None else -2.0)
    return scores


def reward_triggers_in_enum(completions, **kwargs):
    valid = {t.value for t in Trigger}
    scores = []
    for c in completions:
        parsed = _parse_response(c[0]["content"])
        if parsed is None:
            scores.append(-1.0)
            continue
        all_ok = all(e.trigger.value in valid for e in parsed.explanations)
        scores.append(1.0 if all_ok else -1.0)
    return scores

**Reward 2: Trigger correctness (main signal)**

F1 between the set of triggers the model returns and the ground-truth set from the generator. Scaled to roughly [-2, +10] so it dominates the signal.

In [ ]:
def _f1(pred: set, gt: set) -> float:
    if not pred and not gt:
        return 1.0
    if not pred or not gt:
        return 0.0
    tp = len(pred & gt)
    if tp == 0:
        return 0.0
    prec = tp / len(pred)
    rec = tp / len(gt)
    return 2 * prec * rec / (prec + rec)


def reward_triggers_match_ground_truth(completions, ground_truth_triggers, **kwargs):
    scores = []
    for c, gt_list in zip(completions, ground_truth_triggers):
        parsed = _parse_response(c[0]["content"])
        gt = set(gt_list)
        if parsed is None:
            # No parseable output — empty prediction set on non-empty GT.
            scores.append(-2.0 if gt else 0.0)
            continue
        pred = {e.trigger.value for e in parsed.explanations}
        f1 = _f1(pred, gt)
        # Scale F1 in [0, 1] → reward in [-2, +10].
        scores.append(-2.0 + 12.0 * f1)
    return scores

**Rewards 3-5: Production failure categories**

Three verifiable checks lifted directly from the LangSmith error-analysis reports in `.error_analysis_cache/`:

- `reward_previous_dd_amount_correct`: if the text cites a "previous / before / was" £ amount, it must equal `latest.dd_amount - latest.dd_amount_change`. (Failure category 1: 100% of faithfulness failures in one window.)
- `reward_no_hallucinated_facts`: any cited tariff name must appear in the input `contract.tariff_name`s, and any cited rate % must match a `change_since_previous_rate_percent` within ±0.5. (Failure category 2.)
- `reward_underpayment_language_constrained`: "underpayment" language is only allowed when a prior DD change has `is_amount_manually_reduced_lower_than_recommended_amount=True`. (Failure category 3.)

In [ ]:
_PREV_AMOUNT_RE = re.compile(
    r"\b(?:previous|before|was|prior|old)[^.!?\n]{0,60}?£\s*(\d+(?:\.\d{1,2})?)",
    re.IGNORECASE,
)
_TARIFF_RE = re.compile(r"(?:tariff|contract|plan)\s+(?:called\s+|named\s+)?['\"]?([A-Z][A-Za-z0-9 &-]{2,40})['\"]?")
_PERCENT_RE = re.compile(r"(-?\d+(?:\.\d+)?)\s*%")
_UNDERPAY_RE = re.compile(r"\bunderpa(?:y|id|ying|yment)\b", re.IGNORECASE)


def _extract_text(parsed: Optional[DirectDebitExplainerResponse]) -> str:
    if parsed is None:
        return ""
    return " ".join(f"{e.header} {e.explanation}" for e in parsed.explanations)


def reward_previous_dd_amount_correct(completions, input_json, **kwargs):
    scores = []
    for c, inp in zip(completions, input_json):
        text = _extract_text(_parse_response(c[0]["content"]))
        latest = inp["latest_dd_change"]
        expected = round(float(latest["dd_amount"]) - float(latest.get("dd_amount_change") or 0.0), 2)
        cited = [float(m.group(1)) for m in _PREV_AMOUNT_RE.finditer(text)]
        if not cited:
            # Didn't cite anything — neutral, not penalised (we penalise only wrong citations).
            scores.append(0.0)
            continue
        if all(abs(v - expected) <= 0.01 for v in cited):
            scores.append(2.0)
        else:
            scores.append(-3.0)
    return scores


def reward_no_hallucinated_facts(completions, input_json, **kwargs):
    scores = []
    for c, inp in zip(completions, input_json):
        text = _extract_text(_parse_response(c[0]["content"]))
        if not text:
            scores.append(0.0)
            continue
        tariffs = {ch.get("tariff_name", "").lower() for ch in inp["account_context"].get("contract_history", [])}
        # Gather all rate change percents from the current contract(s).
        real_pcts = []
        for ch in inp["account_context"].get("contract_history", []):
            for rh in ch.get("contract_rates_history", []):
                for rate in rh.get("rates", []):
                    v = rate.get("change_since_previous_rate_percent")
                    if v is not None:
                        real_pcts.append(float(v))

        score = 1.0
        # Tariff name citations: any cited name must be a substring of a real tariff (case-insensitive).
        for m in _TARIFF_RE.finditer(text):
            cited = m.group(1).strip().lower()
            if not any(cited in t or t in cited for t in tariffs if t):
                score = -3.0
                break

        # Percent citations: each cited |pct| > 1.0 should be within ±0.5 of a real rate-change pct.
        if score > 0:
            for m in _PERCENT_RE.finditer(text):
                cited_pct = float(m.group(1))
                if abs(cited_pct) < 1.0:
                    continue  # tiny values are often aspirational, ignore
                if not any(abs(cited_pct - rp) <= 0.5 for rp in real_pcts):
                    score = -3.0
                    break

        scores.append(score)
    return scores


def reward_underpayment_language_constrained(completions, input_json, **kwargs):
    scores = []
    for c, inp in zip(completions, input_json):
        text = _extract_text(_parse_response(c[0]["content"]))
        if not _UNDERPAY_RE.search(text):
            scores.append(0.5)  # Not using the language at all is fine.
            continue
        prior_manual = any(
            ch.get("is_amount_manually_reduced_lower_than_recommended_amount", False)
            for ch in inp["account_context"].get("dd_change_history", [])
        )
        scores.append(0.5 if prior_manual else -1.5)
    return scores

**Reward 6: Well-formed explanations**

Each explanation is 1-3 sentences, each header ≤ 10 words. Light shape check to keep output tidy.

In [ ]:
_SENTENCE_SPLIT_RE = re.compile(r"[.!?]+")


def reward_explanations_well_formed(completions, **kwargs):
    scores = []
    for c in completions:
        parsed = _parse_response(c[0]["content"])
        if parsed is None or not parsed.explanations:
            scores.append(-0.5)
            continue
        ok = True
        for e in parsed.explanations:
            if len(e.header.split()) > 10:
                ok = False
                break
            n_sentences = sum(1 for s in _SENTENCE_SPLIT_RE.split(e.explanation) if s.strip())
            if not (1 <= n_sentences <= 3):
                ok = False
                break
        scores.append(0.5 if ok else -0.5)
    return scores


# Quick self-check — score a hand-written good response + a bad one.
_good = """```json
{"explanations": [
  {"trigger": "Change in usage",
   "header": "Your usage went up",
   "explanation": "Your projected annual electricity use climbed by 12%, which means more energy to pay for. Your direct debit has been updated so you cover that higher use. Nothing else has changed."}
]}
```"""
_bad = "Here is some text with no JSON."

_fake = [[{"content": _good}], [{"content": _bad}]]
_fake_gt = [["Change in usage"], ["Change in usage"]]
_fake_inp = [example.model_dump(mode="json"), example.model_dump(mode="json")]
print("schema_valid:", reward_schema_valid(_fake))
print("in_enum    :", reward_triggers_in_enum(_fake))
print("gt_match   :", reward_triggers_match_ground_truth(_fake, _fake_gt))
print("prev_amount:", reward_previous_dd_amount_correct(_fake, _fake_inp))
print("no_halluc  :", reward_no_hallucinated_facts(_fake, _fake_inp))
print("underpay   :", reward_underpayment_language_constrained(_fake, _fake_inp))
print("well_formed:", reward_explanations_well_formed(_fake))

# Dataset Preparation

Create the training dataset.

In [ ]:
from pathlib import Path
from datasets import Dataset

DATA_DIR = Path("/workspace/gemma4_rl/data")
jsonl_files = sorted(DATA_DIR.glob("dd_dataset_*_*rows.jsonl"))

if jsonl_files:
    dataset_path = jsonl_files[-1]  # newest by UTC timestamp in filename
    dataset = Dataset.from_json(str(dataset_path))
    # Line 0 is a metadata record marked with "__meta__": True; drop it.
    if "__meta__" in dataset.column_names:
        dataset = dataset.filter(lambda r: r.get("__meta__") is not True).remove_columns("__meta__")
    if "row_index" in dataset.column_names:
        dataset = dataset.remove_columns("row_index")
    print(f"Loaded {len(dataset)} rows from {dataset_path.name}")
    meta_path = dataset_path.with_suffix(".meta.json")
    if meta_path.exists():
        import json as _json
        meta = _json.loads(meta_path.read_text())
        print(f"  generator_version={meta.get(\"generator_version\")}  seed={meta.get(\"seed\")}  generated_at_utc={meta.get(\"generated_at_utc\")}")
else:
    print("No cached JSONL found; generating in memory via build_dataset(n=1000).")
    rows = build_dataset(n=1000, seed=42)
    dataset = Dataset.from_list(rows)

maximum_length = len(tokenizer.apply_chat_template(
    dataset[0]["prompt"],
    add_generation_prompt=True,
))
print(f"Rows: {len(dataset)}")
print(f"Maximum prompt length (tokens): {maximum_length}")
print(f"Sample ground_truth_triggers: {dataset[0][\"ground_truth_triggers\"]}")


<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations! We also support GSPO, GAPO, Dr GRPO and more! Go the Unsloth [Reinforcement Learning Docs](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide) for more options.

In [ ]:
# Prompt is ~3.1k tokens (domain knowledge + input JSON). With max_seq_length=6144 we have ~3k tokens for completions.
# A parseable DirectDebitExplainerResponse with 1-2 trigger explanations typically fits in 400-900 tokens;
# cap at 1536 for safety (mask_truncated_completions=True below drops anything longer).
max_completion_length = 1536

from trl import GRPOConfig, GRPOTrainer
# A100 40GB sizing — same as before:
#   per_device_bs=8, grad_accum=1, num_generations=4  →  8 sequences per optimizer step.
#   (per_device_bs * grad_accum) must be divisible by num_generations. 8*1=8, divisible by 4. ✓
training_args = GRPOConfig(
    temperature = 1.0,
    learning_rate = 5e-5,
    weight_decay = 0.001,
    warmup_steps = 30,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 8,
    gradient_accumulation_steps = 1,
    num_generations = 4,
    max_completion_length = max_completion_length,
    max_steps = 300,
    save_steps = 50,
    report_to = "none",
    output_dir = "outputs",
    epsilon = 0.2,
    epsilon_high = 0.28,
    delta = 1.5,
    loss_type = 'bnpo',
    mask_truncated_completions = True,
)

And let's run the trainer! Watch the `reward` column — the main component is `reward_triggers_match_ground_truth` (range ≈ -2 to +10). Early on you'll see `reward_schema_valid` punished (the base Gemma 4 E2B rarely produces valid JSON on first try); the shape rewards lift as the model learns the output contract.

Rough expectations on A100 40GB:
- Steps 1-30: shape rewards dominate (schema_valid -2 → +1, explanations_well_formed oscillates). Trigger F1 ≈ 0.
- Steps 30-150: shape stabilises, F1 starts moving.
- Steps 150-300: F1 climbs toward 0.6-0.8; production-failure rewards (previous_amount, no_hallucinated_facts, underpayment) transition from mostly neutral to positive.

In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        reward_schema_valid,
        reward_triggers_in_enum,
        reward_triggers_match_ground_truth,
        reward_previous_dd_amount_correct,
        reward_no_hallucinated_facts,
        reward_underpayment_language_constrained,
        reward_explanations_well_formed,
    ],
    args = training_args,
    train_dataset = dataset,
)

And let's train the model!

On A100 40GB with the settings above, expect ~2–3 seconds per step early on (before completions get long), and the reward signal to start rising somewhere after step ~100. If you hit OOM, lower `per_device_train_batch_size` and `num_generations` to 2.

In [ ]:
trainer.train()

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
model.save_pretrained("gemma_4_lora")  # Local saving
tokenizer.save_pretrained("gemma_4_lora")

Verify LoRA is actually trained!

In [ ]:
from safetensors import safe_open

tensors = {}
with safe_open("gemma_4_lora/adapter_model.safetensors", framework = "pt") as f:
    # Verify both A and B are non zero
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

# Regression against real failure traces

The LangSmith online evaluator flagged 187 prompts in one 7-day window (35% of 704) as faithfulness failures. We have them in `.error_analysis_cache/20260413T075447Z_20260420T075447Z/traces.parquet`. Re-score both the base and GRPO-tuned model on those same prompts using our reward functions — the headline number is how many now pass `reward_previous_dd_amount_correct` and `reward_no_hallucinated_facts`.

In [ ]:
import pandas as pd, torch, json as _json
from pathlib import Path

trace_dir = Path("/workspace/gemma4_rl/.error_analysis_cache/20260413T075447Z_20260420T075447Z")
df = pd.read_parquet(trace_dir / "traces.parquet")
failed = df[df["feedback.direct_debit_faithfulness"] < 1.0].copy()
print(f"Loaded {len(failed)} previously-failed prompts")

# Traces store the transformed input under `outputs.dd_account_context.*`.
# The history columns are JSON-encoded strings — we must json.loads them.

_LATEST_FIELDS = [
    "dd_amount", "dd_amount_change", "recommended_dd_amount",
    "yearly_predicted_energy_cost_gbp", "reason_for_DD_change", "description",
    "collectionDay", "datetime_from", "datetime_to", "is_currently_active_DD",
    "is_exemption", "exemption_expiry_date",
    "is_amount_manually_reduced_lower_than_recommended_amount",
]
_PCH_FUEL_FIELDS = [
    "change_kwh", "change_percent",
    "latest_projected_annual_consumption_kwh", "latest_projection_date",
    "previous_projected_annual_consumption_kwh", "previous_projection_date",
]


def _scalar(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except (TypeError, ValueError):
        pass
    return v


def _json_list(v):
    """Parquet stores history arrays as JSON strings. Return a Python list or []."""
    v = _scalar(v)
    if v is None:
        return []
    if isinstance(v, (list, tuple)):
        return list(v)
    if isinstance(v, str):
        try:
            parsed = _json.loads(v)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []
    return []


def _reconstruct_pin(row):
    try:
        latest = {f: _scalar(row.get(f"outputs.dd_account_context.latest_dd_change.{f}")) for f in _LATEST_FIELDS}
        if latest.get("dd_amount") is None:
            return None

        pch_any = False
        pch = {}
        for fuel in ("electricity", "gas"):
            fuel_dict = {}
            for f in _PCH_FUEL_FIELDS:
                val = _scalar(row.get(f"outputs.dd_account_context.account_context.projected_consumption_history.{fuel}.{f}"))
                if val is not None:
                    fuel_dict[f] = val
            if fuel_dict:
                pch[fuel] = fuel_dict
                pch_any = True

        ctx = {
            "dd_change_history": _json_list(row.get("outputs.dd_account_context.account_context.dd_change_history")),
            "payment_history":   _json_list(row.get("outputs.dd_account_context.account_context.payment_history")),
            "contract_history":  _json_list(row.get("outputs.dd_account_context.account_context.contract_history")),
            "projected_consumption_history": pch if pch_any else None,
        }
        return DDExplainerPromptInput.model_validate({"account_context": ctx, "latest_dd_change": latest})
    except Exception:
        return None


@torch.no_grad()
def _generate_response(pin) -> str:
    text = tokenizer.apply_chat_template(build_chat_messages(pin), tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(images=None, text=text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=1024, temperature=0.0, do_sample=False, use_cache=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


def _score_row(pin, completion_text):
    fake = [[{"content": completion_text}]]
    gt = [sorted(t.value for t in detect_triggers(pin))]
    inp = [pin.model_dump(mode="json")]
    return {
        "schema_valid":          reward_schema_valid(fake)[0],
        "in_enum":               reward_triggers_in_enum(fake)[0],
        "f1_triggers":           reward_triggers_match_ground_truth(fake, gt)[0],
        "prev_amount_correct":   reward_previous_dd_amount_correct(fake, inp)[0],
        "no_hallucinated_facts": reward_no_hallucinated_facts(fake, inp)[0],
        "underpayment_ok":       reward_underpayment_language_constrained(fake, inp)[0],
        "well_formed":           reward_explanations_well_formed(fake)[0],
    }


# Filter to rows whose rendered prompt fits the model's context with room for
# a completion. Real traces can have 30+ prior DD changes and overflow otherwise.
# Target: prompt tokens <= max_seq_length - max_completion_length.
def _prompt_tokens(pin):
    text = tokenizer.apply_chat_template(build_chat_messages(pin), tokenize=False, add_generation_prompt=True)
    return len(tokenizer(text=text, add_special_tokens=False)["input_ids"])


_BUDGET = max_seq_length - max_completion_length - 64
candidates = []
for _, row in failed.iterrows():
    pin = _reconstruct_pin(row)
    if pin is None:
        continue
    if _prompt_tokens(pin) <= _BUDGET:
        candidates.append(pin)
print(f"Candidates after token-length filter: {len(candidates)} / {len(failed)}")

# Subsample to 20 rows for a fast regression pass; raise for a fuller view.
N_REGRESS = 20
rows_out = []
for pin in candidates[:N_REGRESS]:
    completion = _generate_response(pin)
    rows_out.append(_score_row(pin, completion))

regression_df = pd.DataFrame(rows_out)
print(f"Scored {len(regression_df)} rows on the tuned model (skipped {skipped} un-reconstructable).")
print(regression_df.mean(numeric_only=True).round(3).to_string())


<a name="Inference"></a>
# Inference
Now let's try the model we just trained!

In [ ]:
# Roll out the tuned model on a fresh DD example and confirm the output parses.
import random
rng_demo = random.Random(2026)
demo_target = {Trigger.Change_in_unit_rates}
demo_example = generate_dd_example(demo_target, rng_demo)

text = tokenizer.apply_chat_template(
    build_chat_messages(demo_example),
    tokenize = False,
    add_generation_prompt = True,
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(images=None, text=text, add_special_tokens=False, return_tensors="pt").to("cuda"),
    temperature = 0.0,
    do_sample = False,
    max_new_tokens = 1024,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
    use_cache = True,
)

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("gemma_4_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/gemma_4_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("gemma_4_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/gemma_4_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("gemma_4_lora")
    tokenizer.save_pretrained("gemma_4_lora")
if False:
    model.push_to_hub("HF_USERNAME/gemma_4_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/gemma_4_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("gemma_4_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/gemma_4_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("gemma_4_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/gemma_4_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("gemma_4_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/gemma_4_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/gemma_4_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN",
    )

Now, use the `gemma_4_finetune.Q8_0.gguf` file or `gemma_4_finetune.Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).